In [0]:
%sql SHOW TABLES IN workspace.silver

# V-Energy Grid 360 — Camada Gold (DE-03)

## 1. Visão Geral e Arquitetura da Camada Gold
A camada **Gold** representa o topo da arquitetura Medalhão. Sua responsabilidade é consolidar, agregar e enriquecer os dados limpos vindos da camada **Silver**, transformando tabelas relacionais em **visões analíticas de negócio**.

### Objetivos do Negócio Atendidos:
1. **Diagnóstico de Receita e Tarifa:** Responder à diretoria sobre a variação do volume faturado (-4%) versus o reajuste tarifário (+10,5%), identificando o impacto do consumo, bandeiras e Geração Distribuída (GD).
2. **Qualidade do Serviço (DEC/FEC):** Calcular os indicadores regulatórios da ANEEL por alimentador e município para controle operacional.
3. **Fila de Inspeção e Risco:** Identificar potenciais perdas não técnicas (fraudes/gatos) cruzando dados de consumo, inspeções anteriores e chamados do call center.
4. **Visão 360° da Unidade Consumidora:** Fonte única de verdade para alimentar o backend do Portal Web e o Agente de IA.

---

### Histórias de Usuário Atendidas neste Notebook:
* **Modelagem de Métricas:** *"Como engenheiro de dados, quero modelar a camada Gold a partir das perguntas de negócio das lideranças..."*
* **Indicadores ANEEL:** *"Como engenheiro de dados, quero calcular DEC e FEC por alimentador conforme a definição da ANEEL..."*
* **Testes de Qualidade:** *"Como engenheiro de dados, quero testes de qualidade automatizados no pipeline..."*

In [0]:
#secao 0 
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA_SILVER = "silver_publicada"  # ajuste ao schema efetivamente publicado mudar para silver_publicada dps 
SCHEMA_GOLD = "gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_GOLD}")


def assert_no_nulls(df, columns, name):
    """Falha se houver nulos nas colunas informadas."""
    condition = None

    for column in columns:
        part = F.col(column).isNull()
        condition = part if condition is None else (condition | part)

    if condition is not None and df.filter(condition).limit(1).count() > 0:
        raise ValueError(
            f"{name}: há valores nulos nas colunas obrigatórias {columns}."
        )


def assert_unique(df, columns, name):
    """Falha se a chave informada não for única."""
    duplicates = (
        df.groupBy(*columns)
          .count()
          .filter(F.col("count") > 1)
          .limit(1)
          .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"{name}: a chave {columns} não é única. "
            "Revise a granularidade da fonte antes de continuar."
        )


def assert_columns_exist(df, columns, name):
    missing = sorted(set(columns) - set(df.columns))

    if missing:
        raise ValueError(f"{name}: colunas ausentes: {missing}")


def get_distinct_uc_with_gd(df_gd):
    """
    Retorna uma linha por UC com GD.

    A regra considera que qualquer registro em geracao_distribuida
    significa que a UC possui GD. Se a fonte tiver status ou vigência,
    substitua essa regra pela condição de negócio apropriada.
    """
    assert_columns_exist(df_gd, ["id_uc"], "geracao_distribuida")

    return (
        df_gd
        .filter(F.col("id_uc").isNotNull())
        .select("id_uc")
        .dropDuplicates(["id_uc"])
        .withColumn("possui_gd", F.lit("Sim"))
    )


print(f"Silver: {CATALOG}.{SCHEMA_SILVER}")
print(f"Gold:   {CATALOG}.{SCHEMA_GOLD}")

## 2. Diagnóstico de Receita e Variação Tarifária
### Objetivo:
Atender ao pedido do Diretor Comercial explicando a queda no volume faturado e o impacto na receita por município, classe de consumo, subgrupo e presença de geração solar (GD).

### Regras de Negócio e Agregações:
* Cruzamento das tabelas `silver.faturas`, `silver.leituras`, `silver.unidades_consumidoras` e `silver.geracao_distribuida`.
* Métricas geradas: Consumo total ($kWh$), Receita total ($R\$$), Ticket médio e Tarifa Média Efetiva ($R\$/kWh$).

In [0]:
# Inspecionar as colunas da tabela unidades_consumidoras
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras").printSchema()

In [0]:

# SEÇÃO 1: Construção da Gold Diagnóstico de Receita

df_faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
df_leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
df_ucs = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")
df_gd = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida")

# Validar colunas esperadas nas fontes
assert_columns_exist(df_faturas, ["id_uc", "mes_referencia", "valor_total"], "faturas")
# AJUSTADO: mes_referencia no lugar de competencia
assert_columns_exist(df_leituras, ["id_uc", "mes_referencia", "consumo_kwh"], "leituras")
assert_columns_exist(df_ucs, ["id_uc", "municipio", "classe", "subgrupo_tarifario"], "unidades_consumidoras")

# Base de UCs (uma linha de cadastro por UC)
df_ucs_base = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .select("id_uc", "municipio", "classe", "subgrupo_tarifario")
)

assert_unique(df_ucs_base, ["id_uc"], "unidades_consumidoras")

# Agregar faturas por UC e mês
df_faturas_uc_mes = (
    df_faturas
    .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("valor_total").alias("receita_uc_mes"),
        F.count(F.lit(1)).alias("qtd_faturas_uc_mes")
    )
)

assert_unique(df_faturas_uc_mes, ["id_uc", "mes_referencia"], "faturas agregadas por UC e competência")

# Agregar leituras por UC e mês (AJUSTADO: usando mes_referencia)
df_leituras_uc_mes = (
    df_leituras
    .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
    .groupBy("id_uc", "mes_referencia")
    .agg(
        F.sum("consumo_kwh").alias("consumo_uc_mes"),
        F.count(F.col("consumo_kwh")).alias("qtd_leituras_validas_uc_mes")
    )
)

assert_unique(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "leituras agregadas por UC e competência")

# Unir faturas com leituras e cadastro
df_uc_mes = (
    df_faturas_uc_mes
    .join(df_leituras_uc_mes, ["id_uc", "mes_referencia"], "inner")
    .join(df_ucs_base, "id_uc", "left")
)

# Tratar bandeira tarifária se existir em faturas
if "bandeira_tarifaria" in df_faturas.columns:
    df_bandeira_uc_mes = (
        df_faturas
        .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
        .groupBy("id_uc", "mes_referencia")
        .agg(
            F.first("bandeira_tarifaria", ignorenulls=True).alias("bandeira_tarifaria"),
            F.countDistinct("bandeira_tarifaria").alias("qtd_bandeiras_uc_mes")
        )
    )

    df_uc_mes = df_uc_mes.join(df_bandeira_uc_mes, ["id_uc", "mes_referencia"], "left")

    if df_bandeira_uc_mes.filter(F.col("qtd_bandeiras_uc_mes") > 1).limit(1).count() > 0:
        raise ValueError(
            "Há mais de uma bandeira tarifária por UC/competência. "
            "Defina a regra de rateio ou a granularidade correta antes de gerar diagnostico_receita."
        )
else:
    df_uc_mes = df_uc_mes.withColumn("bandeira_tarifaria", F.lit(None).cast("string"))

# Adicionar Flag de GD
df_gd_uc = get_distinct_uc_with_gd(df_gd)

df_uc_mes = (
    df_uc_mes
    .join(df_gd_uc, "id_uc", "left")
    .withColumn("possuo_gd", F.coalesce(F.col("possui_gd"), F.lit("Não")))
    .drop("possui_gd")
)

# Métricas agregadas no grão de negócio
group_columns = [
    "mes_referencia",
    "municipio",
    "classe",
    "subgrupo_tarifario",
    "bandeira_tarifaria",
    "possuo_gd"
]

gold_diagnostico_receita = (
    df_uc_mes
    .groupBy(*group_columns)
    .agg(
        F.countDistinct("id_uc").alias("qtd_ucs_faturadas"),
        F.sum("consumo_uc_mes").alias("consumo_total_kwh"),
        F.sum("receita_uc_mes").alias("receita_total_reais"),
        F.avg("receita_uc_mes").alias("ticket_medio_reais_por_uc"),
        F.avg("consumo_uc_mes").alias("consumo_medio_kwh_por_uc")
    )
    .withColumn(
        "tarifa_media_efetiva",
        F.when(
            F.col("consumo_total_kwh").isNotNull() & (F.col("consumo_total_kwh") != 0),
            F.col("receita_total_reais") / F.col("consumo_total_kwh")
        )
    )
    .withColumnRenamed("mes_referencia", "competencia")
    .withColumnRenamed("classe", "classe_consumo")
)

# Checagens antes da gravação
assert_no_nulls(gold_diagnostico_receita, ["competencia"], "diagnostico_receita")

if gold_diagnostico_receita.filter((F.col("receita_total_reais") < 0) | (F.col("consumo_total_kwh") < 0)).limit(1).count() > 0:
    raise ValueError(
        "diagnostico_receita: foram encontrados valores negativos em receita ou consumo. Revise a regra para estornos e ajustes."
    )

(
    gold_diagnostico_receita
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
)

print(" Tabela 'diagnostico_receita' gerada e gravada com sucesso!")

## 3. Indicadores Operacionais e Regulação ANEEL (DEC / FEC)
### Objetivo:
Calcular a **Duração Equivalente de Interrupção por Unidade Consumidora (DEC)** e a **Frequência Equivalente de Interrupção por Unidade Consumidora (FEC)** agregadas por Alimentador e Município.

### Definição Oficial (PRODIST Módulo 8 / ANEEL):
* **DEC:** $\frac{\text{Soma das Horas de Interrupção}}{\text{Total de UCs do Alimentador}}$
* **FEC:** $\frac{\text{Soma das Ocorrências de Interrupção}}{\text{Total de UCs do Alimentador}}$

In [0]:
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede").printSchema()

In [0]:
# SEÇÃO 2: Construção da Gold DEC e FEC

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "data_abertura", "duracao_min"],
    "ocorrencias_rede"
)

assert_columns_exist(
    df_ucs,
    ["id_uc", "alimentador", "municipio"],
    "unidades_consumidoras"
)

# Uma linha cadastral por UC é necessária para atribuir alimentador e município.
# Se unidades_consumidoras for histórica, selecione o registro vigente
# para cada UC antes desta validação.
df_ucs_rede = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .select(
        "id_uc",
        F.col("alimentador").alias("id_alimentador"),
        "municipio"
    )
)

assert_unique(df_ucs_rede, ["id_uc"], "cadastro de UCs para DEC/FEC")

assert_no_nulls(
    df_ucs_rede,
    ["id_alimentador"],
    "cadastro de UCs para DEC/FEC"
)

# Município é mantido como atributo do alimentador, não como dimensão
# que alteraria a soma e o denominador do indicador.
df_municipio_alimentador = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("municipio").alias("qtd_municipios"),
        F.first("municipio", ignorenulls=True).alias("municipio")
    )
)

if (
    df_municipio_alimentador
    .filter(F.col("qtd_municipios") > 1)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "Há alimentadores associados a mais de um município. "
        "Defina uma regra para o município exibido ou remova essa "
        "dimensão da tabela de indicadores."
    )

# Validar campos essenciais e duração.
df_ocorrencias_validas = (
    df_ocorrencias
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & F.col("data_abertura").isNotNull()
    )
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "ocorrencias_rede contém durações negativas."
    )

# Cada evento por UC deve aparecer uma única vez.
assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

df_ocorrencias_prep = (
    df_ocorrencias_validas
    .withColumn(
        "competencia",
        F.date_format(F.to_timestamp("data_abertura"), "yyyy-MM")
    )
    .withColumn(
        "duracao_horas",
        F.col("duracao_min").cast("double") / F.lit(60.0)
    )
)

# Associar alimentador à ocorrência. O cadastro já foi validado
# como uma linha por UC.
df_ocorrencias_rede = (
    df_ocorrencias_prep
    .join(df_ucs_rede, "id_uc", "inner")
)

# Numerador por alimentador e competência:
# duração acumulada e número de eventos UC-ocorrência.
df_numeradores = (
    df_ocorrencias_rede
    .groupBy("id_alimentador", "competencia")
    .agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(
            F.struct("id_uc", "id_ocorrencia")
        ).alias("frequencia_total_interrupcoes"),
        F.sum("duracao_horas")
         .alias("duracao_total_interrupcoes_horas")
    )
)

# Denominador cadastral atual por alimentador.
df_denominadores = (
    df_ucs_rede
    .groupBy("id_alimentador")
    .agg(
        F.countDistinct("id_uc").alias("total_ucs_alimentador")
    )
)

gold_indicadores_dec_fec = (
    df_numeradores
    .join(df_denominadores, "id_alimentador", "left")
    .join(df_municipio_alimentador.select("id_alimentador", "municipio"),
          "id_alimentador", "left")
    .withColumn(
        "DEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("duracao_total_interrupcoes_horas")
            / F.col("total_ucs_alimentador")
        )
    )
    .withColumn(
        "FEC",
        F.when(
            F.col("total_ucs_alimentador") > 0,
            F.col("frequencia_total_interrupcoes")
            / F.col("total_ucs_alimentador")
        )
    )
    .select(
        "id_alimentador",
        "municipio",
        "competencia",
        "total_ucs_alimentador",
        "ucs_interrompidas",
        "frequencia_total_interrupcoes",
        "duracao_total_interrupcoes_horas",
        "DEC",
        "FEC"
    )
)

assert_no_nulls(
    gold_indicadores_dec_fec,
    ["id_alimentador", "competencia", "total_ucs_alimentador"],
    "indicadores_dec_fec"
)

if (
    gold_indicadores_dec_fec
    .filter(
        (F.col("total_ucs_alimentador") <= 0)
        | (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec: denominador inválido ou métrica negativa."
    )

(
    gold_indicadores_dec_fec
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
    )
)

print("Tabela indicadores_dec_fec gravada.")

## 4. Fila de Inspeção e Risco de Perdas Não Técnicas
### Objetivo:
Consolidar indicadores de risco e anomalia de consumo por Unidade Consumidora para priorização das equipes de campo no combate a fraudes e furtos.

### Regra de Priorização:
* **ALTA - REINCIDENTE:** UCs com histórico prévio de irregularidades confirmadas.
* **ALTA - CONSUMO ZERO:** UCs sem geração solar conectada que apresentaram consumo zerado no histórico.
* **MÉDIA - NUNCA INSPECCIONADO:** UCs que jamais receberam fiscalização de campo.
* **BAIXA:** Demais UCs operando em padrão regular.

In [0]:
print("--- SCHEMA: inspecoes ---")
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.inspecoes").printSchema()

print("\n--- SCHEMA: atendimentos ---")
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.atendimentos").printSchema()

print("\n--- SCHEMA: leituras ---")
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras").printSchema()

In [0]:
# SEÇÃO 3: Construção da Gold Fila de Inspeção

df_inspecoes = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.inspecoes"
)

df_atendimentos = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.atendimentos"
)

df_leituras = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.leituras"
)

df_ucs = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras"
)

df_gd = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.geracao_distribuida"
)

assert_columns_exist(
    df_inspecoes,
    ["id_uc", "id_inspecao", "resultado"],
    "inspecoes"
)

assert_columns_exist(
    df_atendimentos,
    ["id_uc", "protocolo", "nota_satisfacao"],
    "atendimentos"
)

assert_columns_exist(
    df_leituras,
    ["id_uc", "consumo_kwh"],
    "leituras"
)

assert_columns_exist(
    df_ucs,
    ["id_uc"],
    "unidades_consumidoras"
)

# Uma linha por UC na base da fila.
df_ucs_base = (
    df_ucs
    .filter(F.col("id_uc").isNotNull())
    .dropDuplicates(["id_uc"])
)

assert_unique(df_ucs_base, ["id_uc"], "base da fila de inspeção")

# Validar que cada ID de inspeção identifica uma única inspeção.
df_inspecoes_validas = df_inspecoes.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_inspecao").isNotNull()
)

assert_unique(
    df_inspecoes_validas,
    ["id_inspecao"],
    "inspecoes por id_inspecao"
)

df_insp_summary = (
    df_inspecoes_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_inspecao")
         .alias("total_inspecoes_realizadas"),
        F.countDistinct(
            F.when(
                F.col("resultado") == "Irregular",
                F.col("id_inspecao")
            )
        ).alias("total_inspecoes_irregulares")
    )
)

# Contar protocolos distintos para evitar duplicidade por linha.
df_atend_validos = df_atendimentos.filter(
    F.col("id_uc").isNotNull()
    & F.col("protocolo").isNotNull()
)

df_atend_summary = (
    df_atend_validos
    .groupBy("id_uc")
    .agg(
        F.countDistinct("protocolo")
         .alias("total_chamados_callcenter"),
        F.avg("nota_satisfacao")
         .alias("media_satisfacao_atendimento")
    )
)

# Média do consumo por UC; nulos são ignorados pelo avg.
df_consumo_medio = (
    df_leituras
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("consumo_kwh").isNotNull()
    )
    .groupBy("id_uc")
    .agg(
        F.avg("consumo_kwh")
         .alias("media_consumo_historico")
    )
)

df_gd_uc = get_distinct_uc_with_gd(df_gd)

gold_fila_inspecao = (
    df_ucs_base
    .join(df_consumo_medio, "id_uc", "left")
    .join(df_insp_summary, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(
        df_gd_uc
        .withColumnRenamed("possui_gd", "possui_geracao_solar"),
        "id_uc",
        "left"
    )
    .withColumn(
        "total_inspecoes_realizadas",
        F.coalesce(F.col("total_inspecoes_realizadas"), F.lit(0))
    )
    .withColumn(
        "total_inspecoes_irregulares",
        F.coalesce(F.col("total_inspecoes_irregulares"), F.lit(0))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "possui_geracao_solar",
        F.coalesce(F.col("possui_geracao_solar"), F.lit("Não"))
    )
    .withColumn(
        "prioridade_inspecao",
        F.when(
            F.col("total_inspecoes_irregulares") > 0,
            "ALTA - HISTORICO IRREGULAR"
        )
        .when(
            F.col("media_consumo_historico").isNull(),
            "REVISAR - SEM HISTORICO DE CONSUMO"
        )
        .when(
            (F.col("media_consumo_historico") == 0)
            & (F.col("possui_geracao_solar") == "Não"),
            "ALTA - CONSUMO ZERO"
        )
        .when(
            F.col("total_inspecoes_realizadas") == 0,
            "MEDIA - NUNCA INSPECIONADO"
        )
        .otherwise("BAIXA")
    )
)

assert_unique(
    gold_fila_inspecao,
    ["id_uc"],
    "gold fila_inspecao"
)

assert_no_nulls(
    gold_fila_inspecao,
    ["id_uc", "prioridade_inspecao"],
    "gold fila_inspecao"
)

(
    gold_fila_inspecao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao"
    )
)

print("Tabela fila_inspecao gravada.")

## 5. Visão 360° da Unidade Consumidora (Tabela Fato/Dimensão Central)
### Objetivo:
Criar a tabela unificada que alimentará o **Portal Web** e o **Agente de IA**, permitindo consultar a história completa de qualquer cliente em uma única leitura.

In [0]:

# SEÇÃO 4: Construção da Gold Visão 360

df_ocorrencias = spark.table(
    f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede"
)

assert_columns_exist(
    df_ocorrencias,
    ["id_uc", "id_ocorrencia", "duracao_min"],
    "ocorrencias_rede"
)

# Uma linha por ocorrência sofrida por UC.
df_ocorrencias_validas = df_ocorrencias.filter(
    F.col("id_uc").isNotNull()
    & F.col("id_ocorrencia").isNotNull()
)

assert_unique(
    df_ocorrencias_validas,
    ["id_uc", "id_ocorrencia"],
    "ocorrencias_rede por UC e ocorrência"
)

if (
    df_ocorrencias_validas
    .filter(F.col("duracao_min") < 0)
    .limit(1)
    .count() > 0
):
    raise ValueError("ocorrencias_rede contém durações negativas.")

df_interrupcao_agg = (
    df_ocorrencias_validas
    .groupBy("id_uc")
    .agg(
        F.countDistinct("id_ocorrencia")
         .alias("total_interrupcoes_sofridas"),
        F.sum(F.col("duracao_min").cast("double") / F.lit(60.0))
         .alias("duracao_total_interrupcoes_horas")
    )
)

gold_visao_360 = (
    df_ucs_base
    .join(df_gd_uc, "id_uc", "left")
    .join(df_atend_summary, "id_uc", "left")
    .join(df_interrupcao_agg, "id_uc", "left")
    .join(df_consumo_medio, "id_uc", "left")
    .withColumn(
        "possui_gd",
        F.coalesce(F.col("possui_gd"), F.lit("Não"))
    )
    .withColumn(
        "total_chamados_callcenter",
        F.coalesce(F.col("total_chamados_callcenter"), F.lit(0))
    )
    .withColumn(
        "total_interrupcoes_sofridas",
        F.coalesce(F.col("total_interrupcoes_sofridas"), F.lit(0))
    )
    .withColumn(
        "duracao_total_interrupcoes_horas",
        F.coalesce(
            F.col("duracao_total_interrupcoes_horas"),
            F.lit(0.0)
        )
    )
)

assert_unique(gold_visao_360, ["id_uc"], "gold visao_360_uc")
assert_no_nulls(gold_visao_360, ["id_uc"], "gold visao_360_uc")

(
    gold_visao_360
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
)

print("Tabela visao_360_uc gravada.")

## 6. Testes de Qualidade Automatizados e Validação do Pipeline
### Objetivo:
Garantir que os dados gerados na Gold atendem às restrições de integridade, contagem e consistência lógica antes de serem exportados para o banco de dados local da aplicação.

In [0]:
# SEÇÃO 5: Suíte Completa de Testes Automatizados de Qualidade

from pyspark.sql import functions as F

tables_gold_pk = {
    "diagnostico_receita": [
        "competencia",
        "municipio",
        "classe_consumo",
        "subgrupo_tarifario",
        "bandeira_tarifaria",
        "possuo_gd"
    ],
    "indicadores_dec_fec": [
        "id_alimentador",
        "competencia"
    ],
    "fila_inspecao": ["id_uc"],
    "visao_360_uc": ["id_uc"]
}

# Confere que todas as tabelas existem e têm registros.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_columns_exist(df, pk_cols, nome_completo)

    if df.limit(1).count() == 0:
        raise ValueError(f"{nome_completo} está vazia.")

    print(f"[PASS] {nome_completo}: possui registros.")

# Chaves compostas não podem conter nulos.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_no_nulls(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave sem nulos.")

# A chave declarada deve ser única.
for tabela, pk_cols in tables_gold_pk.items():
    nome_completo = f"{CATALOG}.{SCHEMA_GOLD}.{tabela}"
    df = spark.table(nome_completo)

    assert_unique(df, pk_cols, nome_completo)
    print(f"[PASS] {nome_completo}: chave única.")

# Verificar métricas negativas.
df_dec = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec"
)

if (
    df_dec
    .filter(
        (F.col("DEC") < 0)
        | (F.col("FEC") < 0)
        | (F.col("duracao_total_interrupcoes_horas") < 0)
        | (F.col("frequencia_total_interrupcoes") < 0)
        | (F.col("total_ucs_alimentador") <= 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "indicadores_dec_fec contém valores negativos "
        "ou denominadores inválidos."
    )

# DEC e FEC não devem ser nulos nos grupos gerados.
if (
    df_dec
    .filter(F.col("DEC").isNull() | F.col("FEC").isNull())
    .limit(1)
    .count() > 0
):
    raise ValueError("indicadores_dec_fec contém DEC ou FEC nulos.")

print("[PASS] DEC/FEC sem valores negativos, nulos ou denominadores inválidos.")

# Verificar métricas de receita e consumo.
df_receita = spark.table(
    f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita"
)

if (
    df_receita
    .filter(
        (F.col("receita_total_reais") < 0)
        | (F.col("consumo_total_kwh") < 0)
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita contém receita ou consumo negativos. "
        "Confirme se estornos e ajustes devem ser permitidos."
    )

# A tarifa efetiva pode ser nula quando consumo total é zero ou nulo.
if (
    df_receita
    .filter(
        (F.col("consumo_total_kwh") > 0)
        & F.col("tarifa_media_efetiva").isNull()
    )
    .limit(1)
    .count() > 0
):
    raise ValueError(
        "diagnostico_receita tem tarifa efetiva nula "
        "apesar de consumo positivo."
    )

print("[PASS] Receita e consumo dentro dos limites definidos.")

# Formato estrito YYYY-MM e competência calendário válida.
competencia_invalida = (
    df_receita
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_invalida > 0:
    raise ValueError(
        "diagnostico_receita contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de receita no formato YYYY-MM.")

# Checagem equivalente para DEC/FEC.
competencia_dec_invalida = (
    df_dec
    .filter(
        F.col("competencia").isNull()
        | ~F.col("competencia").rlike(
            r"^[0-9]{4}-(0[1-9]|1[0-2])$"
        )
    )
    .limit(1)
    .count()
)

if competencia_dec_invalida > 0:
    raise ValueError(
        "indicadores_dec_fec contém competência fora do formato YYYY-MM."
    )

print("[PASS] Competências de DEC/FEC no formato YYYY-MM.")

# Garantir que nenhuma UC foi multiplicada nas tabelas de visão/fila.
for tabela in ["fila_inspecao", "visao_360_uc"]:
    df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

    assert_unique(df, ["id_uc"], f"{CATALOG}.{SCHEMA_GOLD}.{tabela}")

print("[PASS] Uma linha por UC na fila e na visão 360.")
print("Todos os testes de qualidade da Gold foram concluídos.")

In [0]:
tables_gold = {
    "Diagnóstico de Receita": f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita",
    "Indicadores DEC/FEC": f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec",
    "Fila de Inspeção": f"{CATALOG}.{SCHEMA_GOLD}.fila_inspecao",
    "Visão 360 da UC": f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc"
}

print("=" * 80)
print("PAINEL DE AUDITORIA E INSPEÇÃO DA CAMADA GOLD")
print("=" * 80)

for nome_exibicao, tabela_full in tables_gold.items():
    df = spark.table(tabela_full)
    total_registros = df.count()
    total_colunas = len(df.columns)

    print(f"\n🔹 TABELA: {nome_exibicao.upper()}")
    print(f"   • Caminho: {tabela_full}")
    print(f"   • Volumetria: {total_registros:,} linhas | {total_colunas} colunas")
    print(f"   • Colunas: {', '.join(df.columns)}")
    print(f"   • Amostra dos dados:")

    display(df.limit(5))

    print("-" * 80)

print("Inspeção da camada Gold concluída.")

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")

print("Total de linhas:", df.count())
print("UCs distintas:", df.select("id_uc").distinct().count())
print("IDs nulos:", df.filter(F.col("id_uc").isNull()).count())

duplicadas = (
    df.groupBy("id_uc")
      .count()
      .filter(F.col("count") > 1)
)

print("IDs de UC duplicados:", duplicadas.count())
display(duplicadas.limit(10))

##  Validação do Pipeline Gold e Reconciliação de Dados (Silver ➔ Gold)

Esta etapa de validação garante a integridade estrutural, a precisão matemática e a rastreabilidade do pipeline da camada **Gold** após a integração com as atualizações da camada **Silver**.

###  Escopo dos Testes Aplicados:
1. **Reconciliação Financeira (Faturas & Consumo):** Compara os totais de receita (`valor_total`) e consumo (`consumo_kwh`) agregados na Gold com a fonte Silver para garantir que os `JOINs` não geraram duplicações de linhas ("efeito leque").
2. **Reconciliação Operacional (DEC/FEC - Ocorrências de Rede):** Recalcula do zero os indicadores de frequência e duração de interrupções por alimentador na Silver e compara diretamente com as métricas consolidadas na Gold.
3. **Invariantes e Métricas Inválidas:** Verifica a ausência de valores nulos (`null`), negativos ou infinitos nas métricas financeiras finais.
4. **Integridade de Cobertura Temporal:** Confirma se todas as competências (mês/ano) tratadas na Silver foram preservadas na Gold.
5. **Consistência de Universo (Visão 360):** Assegura que o universo de Unidades Consumidoras (UCs) na Visão 360 bate com o cadastro ativo.

In [0]:
# 1. TESTE DE RECONCILIAÇÃO FINANCEIRA (FATURAS E CONSUMO) - CORRIGIDO

from pyspark.sql import functions as F

# Carregamento das tabelas
faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")
leituras = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.leituras")
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")

# Agrega faturas na granularidade UC/Mês
faturas_uc_mes = (
    faturas
    .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
    .groupBy("id_uc", "mes_referencia")
    .agg(F.sum("valor_total").alias("receita_uc_mes"))
)

# Agrega leituras na granularidade UC/Mês (Usando mes_referencia)
leituras_uc_mes = (
    leituras
    .filter(F.col("id_uc").isNotNull() & F.col("mes_referencia").isNotNull())
    .groupBy("id_uc", "mes_referencia")
    .agg(F.sum("consumo_kwh").alias("consumo_uc_mes"))
)

# Cruzamento idêntico à regra de negócio da Gold (Inner Join)
base_reconciliacao = faturas_uc_mes.join(
    leituras_uc_mes,
    ["id_uc", "mes_referencia"],
    "inner"
)

# Totais esperados a partir da Silver
esperado = base_reconciliacao.agg(
    F.sum("receita_uc_mes").alias("receita_esperada"),
    F.sum("consumo_uc_mes").alias("consumo_esperado"),
    F.countDistinct("id_uc").alias("ucs_distintas_na_base")
)

# Totais obtidos gravados na Gold
obtido = gold_receita.agg(
    F.sum("receita_total_reais").alias("receita_gold"),
    F.sum("consumo_total_kwh").alias("consumo_gold"),
    F.sum("qtd_ucs_faturadas").alias("ucs_somadas_na_gold")
)

# Tabela de comparação final
comparacao = esperado.crossJoin(obtido).select(
    "receita_esperada",
    "receita_gold",
    (F.col("receita_gold") - F.col("receita_esperada")).alias("diferenca_receita"),
    "consumo_esperado",
    "consumo_gold",
    (F.col("consumo_gold") - F.col("consumo_esperado")).alias("diferenca_consumo"),
    "ucs_distintas_na_base",
    "ucs_somadas_na_gold"
)

print("--- RESULTADO DA RECONCILIAÇÃO FINANCEIRA ---")
display(comparacao)

In [0]:
# 2. TESTE DE RECONCILIAÇÃO OPERACIONAL (DEC/FEC - QUALIDADE DA REDE)

from pyspark.sql import functions as F

ocorrencias_tbl = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.ocorrencias_rede")
ucs_tbl = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.unidades_consumidoras")

alimentador_teste = "ALM-REC-013"
competencia_teste = "2026-01"

print(f"Testando Alimentador: {alimentador_teste} | Competência: {competencia_teste}")

# 1. Filtra as UCs do alimentador desejado (evitando ambiguidades)
ucs_filtro = (
    ucs_tbl
    .select("id_uc", F.col("alimentador").alias("alimentador_uc"))
    .filter(F.col("alimentador_uc") == alimentador_teste)
    .dropDuplicates(["id_uc"])
)

# 2. Junta com ocorrencias e calcula as metricas
base_teste = (
    ocorrencias_tbl
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("id_ocorrencia").isNotNull()
        & (F.date_format(F.col("data_abertura"), "yyyy-MM") == competencia_teste)
    )
    .join(ucs_filtro, "id_uc", "inner")
)

print("\n1. Recalculado do Zero direto da Silver:")
display(
    base_teste.agg(
        F.countDistinct("id_uc").alias("ucs_interrompidas"),
        F.countDistinct(F.struct("id_uc", "id_ocorrencia")).alias("frequencia_recalculada"),
        (F.sum(F.col("duracao_min").cast("double")) / F.lit(60.0)).alias("duracao_horas_recalculada")
    )
)

print("\n2. Métricas gravadas na tabela Gold de DEC/FEC:")
display(
    spark.table(f"{CATALOG}.{SCHEMA_GOLD}.indicadores_dec_fec")
    .filter(
        (F.col("id_alimentador") == alimentador_teste)
        & (F.col("competencia") == competencia_teste)
    )
)

In [0]:
# 3. TESTES COMPLEMENTARES: NULOS, TEMPORAL E VISÃO 360 - CORRIGIDO

from pyspark.sql import functions as F

print("--- RUNNING SANITY CHECKS ---")

# 3.1 Teste de Nulos ou Métricas Inválidas
gold_receita = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.diagnostico_receita")
invalidos = gold_receita.filter(
    F.col("receita_total_reais").isNull() |
    F.col("consumo_total_kwh").isNull() |
    (F.col("receita_total_reais") < 0) |
    (F.col("consumo_total_kwh") < 0)
).count()

print(f"[TESTE 1] Métricas nulas ou inválidas na Gold: {invalidos} (Esperado: 0)")

# 3.2 Cobertura Temporal de Competências (Identificando dinamicamente a coluna certa)
silver_faturas = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.faturas")

col_silver = "competencia" if "competencia" in silver_faturas.columns else "mes_referencia"
col_gold = "competencia" if "competencia" in gold_receita.columns else "mes_referencia"

comp_silver = set(r[col_silver] for r in silver_faturas.select(col_silver).distinct().collect() if r[col_silver])
comp_gold = set(r[col_gold] for r in gold_receita.select(col_gold).distinct().collect() if r[col_gold])

diff_meses = comp_silver - comp_gold
print(f"[TESTE 2] Competências ausentes na Gold: {diff_meses} (Esperado: set())")

# 3.3 Cobertura de UCs na Visão 360
v360 = spark.table(f"{CATALOG}.{SCHEMA_GOLD}.visao_360_uc")
total_v360 = v360.select("id_uc").distinct().count()
print(f"[TESTE 3] Total de UCs únicas na Visão 360: {total_v360}")

if invalidos == 0 and len(diff_meses) == 0:
    print("\n TODOS OS TESTES PASSARAM COM SUCESSO! A CAMADA GOLD ESTÁ HOMOLOGADA.")
else:
    print("\n VERIFICAR DIVERGÊNCIAS ENCONTRADAS NOS TESTES.")